In [1]:
#--------------------------------------------------------------------------------------------#
#
# Code: CC01_CCAR_A01_data_simulation_01.ipynb
#
# Objective: Step 1: To simulate some data set for CCAR modeling
#            Step 2: To calculate monthly amortization payment and update the current_balance column
#
#            Jingru Chen
#            2026-03-18
#
#------------------------------------------------------------------------------------------#

# Step 0: Upload library

In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-18 13:19:46 EDT
2026-03-18 01:19:46 PM EDT


In [4]:
pwd

'/content'

In [5]:
cd /content/sample_data

/content/sample_data


In [6]:
myout= "/content/sample_data"

# Step 1: Simulating CCAR data set

In [7]:
global n_size
n_size=5000
n_month=48

In [8]:
np.random.seed(42)

def simulate_ccar_portfolio(
    n_loans= n_size ,
    start_date=datetime(2018,1,1),
    months=72,               # 6-year projection horizon (typical CCAR)
    vintage_spread_years=5,  # vintages over last 5 years
    retail=True              # retail vs wholesale toggle
):
    """
    Simulate loan-level data with macroeconomic stress path
    for CCAR-style credit loss model development / challenger testing
    """
    dates = pd.date_range(start_date, periods=months+1, freq='M')
    n_periods = len(dates)

    # --- 1. Create base loan characteristics ---
    df = pd.DataFrame({
        'loan_id': np.arange(1, n_loans+1),
        'origination_date': np.random.choice(
            pd.date_range(start_date - timedelta(days=365*vintage_spread_years), start_date, freq='M'),
            size=n_loans
        ),
        'original_balance': np.random.lognormal(mean=11.5, sigma=1.1, size=n_loans).round(2),  # ~$50k–$300k
        'current_balance_bk': lambda x: x * np.random.uniform(0.4, 1.0, n_loans),  # amortizing
        'loan_to_value_orig': np.random.beta(4, 6, n_loans) * 100 + 40,         # 40–140%
        'credit_score_orig': np.random.normal(720, 60, n_loans).clip(300, 850).round(0),
        'interest_rate': np.random.uniform(3.0, 9.0, n_loans).round(3),
        'loan_term_months': np.random.choice([180,240,360], n_loans, p=[0.2,0.3,0.5]),
        'product_type': np.random.choice(['Mortgage','Auto','Card','HELOC'], n_loans, p=[0.4,0.25,0.25,0.1]),
    })

    df['current_balance_bk'] = (df['original_balance'] *
                            np.random.uniform(0.4, 1.0, n_loans)).round(2)

    # --- 2. Macroeconomic scenario path (severe stress example) ---
    macro = pd.DataFrame(index=dates)
    macro['unemployment'] = 4.0 + np.linspace(0, 6.5, n_periods) + np.random.normal(0, 0.4, n_periods)
    macro['gdp_growth_qoq'] = np.concatenate(([0], np.diff(np.cumsum(np.random.normal(-1.2, 1.8, n_periods)))))
    macro['hpi_change'] = np.random.normal(-0.8, 1.5, n_periods).cumsum().clip(-35, 10)
    macro['bbb_spread'] = 1.8 + np.linspace(0, 5.2, n_periods) + np.random.normal(0, 0.6, n_periods)

    # --- 3. Simulate credit performance (simplified Markov chain style) ---
    df['delinquency_status'] = 0
    df['ever_defaulted'] = False
    df['time_in_default'] = 0
    df['default_date'] = pd.NaT

    # Simplified point-in-time PD/LGD/EAD logic (real models are much more complex)
    base_pd = 1 / (1 + np.exp(6 - 0.008*df['credit_score_orig'] + 0.03*df['loan_to_value_orig']))
    sensitivity_unemp = 0.18
    sensitivity_hpi   = -0.009

    for t in range(1, n_periods):
        current_date = dates[t]

        # Only active loans
        active = (~df['ever_defaulted']) & (df['origination_date'] <= current_date)

        # Stressed PD
        pd_t = base_pd[active] * np.exp(
            sensitivity_unemp * (macro.loc[current_date, 'unemployment'] - 4.0) +
            sensitivity_hpi   * (macro.loc[current_date, 'hpi_change'] / 100)
        ).clip(0.0001, 0.40)

        # Simulate default events
        default_this_month = np.random.random(sum(active)) < pd_t
        def_idx = df.index[active][default_this_month]

        df.loc[def_idx, 'ever_defaulted'] = True
        df.loc[def_idx, 'default_date'] = current_date
        df.loc[def_idx, 'time_in_default'] = 1

        # LGD increases in stress (simplified)
        df['lgd'] = 0.35 + 0.25 * (macro['unemployment']/10).mean()   # time-invariant for simplicity

    # --- 4. Final columns typically needed for model development ---
    df['EAD'] = df['current_balance_bk'].round(2)
    df['PD']  = base_pd.round(4)   # baseline; real models produce PIT PD forecasts
    df['LGD'] = np.random.beta(5, 8, n_loans) * 0.8 + 0.1   # 10–90%, skewed high
    df['projected_loss'] = (df['EAD'] * df['PD'] * df['LGD']).round(2)

    # Cumulative loss for portfolio view
    df['cumulative_loss'] = df.groupby('product_type')['projected_loss'].transform('sum')

    # Merge macro (for model training)
    df = df.merge(macro.reset_index().rename(columns={'index':'report_date'}), how='cross')

    # Logic: (Year Diff * 12) + Month Diff
    df['num_payments'] = (
    (df['report_date'].dt.year - df['origination_date'].dt.year) * 12 +
    (df['report_date'].dt.month - df['origination_date'].dt.month)
    )

    cols_of_interest = [
        'loan_id', 'origination_date', 'report_date', 'num_payments',
        'original_balance', 'current_balance_bk', 'EAD',
        'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'product_type', 'ever_defaulted', 'default_date',
        'PD', 'LGD', 'projected_loss', 'loan_term_months',
        'unemployment', 'gdp_growth_qoq', 'hpi_change', 'bbb_spread'
    ]

    return df[cols_of_interest].sort_values(['loan_id', 'report_date'])


# Example usage
df_panel = simulate_ccar_portfolio(n_loans=n_size, months=n_month)
print("------------------01.A: Shape of portfolio file ---------:", df_panel.shape)
print("\n------------------01.B: Column List of portfolio file ---------:", df_panel.info() )
print("\n------------------01.C: Freq Table of ever_defaulted ---------:\n", df_panel.ever_defaulted.value_counts() )
print("\n------------------01.D: Sample rows of portfolio ---------:\n", df_panel.head(8))

# Quick portfolio summary under stress
print("\n------------------01.E: Portfolio stress statistics:")
print( df_panel.groupby('product_type').agg({
    'projected_loss': 'sum',
    'EAD': 'sum',
    'ever_defaulted': 'mean'
}).round(2))

/tmp/ipykernel_2444/4143616877.py:14: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(start_date, periods=months+1, freq='M')
/tmp/ipykernel_2444/4143616877.py:21: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  pd.date_range(start_date - timedelta(days=365*vintage_spread_years), start_date, freq='M'),


------------------01.A: Shape of portfolio file ---------: (245000, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245000 entries, 0 to 244999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   loan_id             245000 non-null  int64         
 1   origination_date    245000 non-null  datetime64[ns]
 2   report_date         245000 non-null  datetime64[ns]
 3   num_payments        245000 non-null  int32         
 4   original_balance    245000 non-null  float64       
 5   current_balance_bk  245000 non-null  float64       
 6   EAD                 245000 non-null  float64       
 7   credit_score_orig   245000 non-null  float64       
 8   loan_to_value_orig  245000 non-null  float64       
 9   interest_rate       245000 non-null  float64       
 10  product_type        245000 non-null  object        
 11  ever_defaulted      245000 non-null  bool          
 12  default_date  

    original_balance: float | pd.Series,
    annual_rate: float | pd.Series,
    original_term_months: int | pd.Series,
    num_payments: int | pd.Series,           # number of payments already completed
    monthly_prepay_cpr: float = 0.0           # constant CPR; can be Series later

In [9]:
print("------------------01.E: Column List of of portfolio---------:", df_panel.columns )
print("\n------------------01.F: Missing of ever_defaulted ---------:", df_panel.default_date.isnull().sum() )
print("------------------01.G: Freq Table of default_date---------:\n",  df_panel.default_date.value_counts() )
print("\n------------------01.H: Freq Table of interest_rate---------:\n", df_panel.interest_rate.value_counts() )

------------------01.E: Column List of of portfolio---------: Index(['loan_id', 'origination_date', 'report_date', 'num_payments',
       'original_balance', 'current_balance_bk', 'EAD', 'credit_score_orig',
       'loan_to_value_orig', 'interest_rate', 'product_type', 'ever_defaulted',
       'default_date', 'PD', 'LGD', 'projected_loss', 'loan_term_months',
       'unemployment', 'gdp_growth_qoq', 'hpi_change', 'bbb_spread'],
      dtype='object')

------------------01.F: Missing of ever_defaulted ---------: 73843
------------------01.G: Freq Table of default_date---------:
 default_date
2018-02-28    7301
2018-03-31    7203
2018-07-31    6909
2018-04-30    6811
2018-05-31    6762
2018-08-31    6517
2018-06-30    5880
2018-09-30    5733
2018-10-31    5586
2018-11-30    5537
2019-01-31    5047
2019-05-31    4606
2019-02-28    4557
2018-12-31    4557
2019-03-31    4459
2019-06-30    4165
2019-07-31    4116
2019-04-30    4116
2019-09-30    3479
2019-08-31    3479
2020-02-29    3283
2020

In [10]:
df_panel.loc[df_panel['loan_id']== 1 ]
# df_panel.loc[df_panel['account_id']= 1][["account_id", "default_date"]]

,loan_id,origination_date,report_date,num_payments,original_balance,current_balance_bk,EAD,credit_score_orig,loan_to_value_orig,interest_rate,...,ever_defaulted,default_date,PD,LGD,projected_loss,loan_term_months,unemployment,gdp_growth_qoq,hpi_change,bbb_spread
0,1,2016-03-31,2018-01-31,22,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,4.191571,0.000000,0.017034,1.237389
1,1,2016-03-31,2018-02-28,23,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,3.713178,-0.220578,0.340586,1.549005
2,1,2016-03-31,2018-03-31,24,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,4.782095,-2.366107,0.444276,1.268521
3,1,2016-03-31,2018-04-30,25,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,5.189047,2.633310,-0.915421,2.296397
4,1,2016-03-31,2018-05-31,26,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,4.904631,-0.405726,0.749689,2.411752
5,1,2016-03-31,2018-06-30,27,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,5.246438,-0.984923,-1.056674,2.991962
6,1,2016-03-31,2018-07-31,28,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,3.717565,-2.291407,-6.024290,2.322780
7,1,2016-03-31,2018-08-31,29,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,5.883795,0.448327,-9.525102,3.198286
8,1,2016-03-31,2018-09-30,30,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,5.041596,-3.381496,-10.500438,3.058415
9,1,2016-03-31,2018-10-31,31,541879.93,430203.61,430203.61,731.0,81.022504,3.174,...,True,2018-07-31,0.0703,0.306242,9261.76,360,6.161137,-0.631611,-11.797755,4.250539


In [11]:
df_panel.shape

(245000, 21)

In [12]:
df_panel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245000 entries, 0 to 244999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   loan_id             245000 non-null  int64         
 1   origination_date    245000 non-null  datetime64[ns]
 2   report_date         245000 non-null  datetime64[ns]
 3   num_payments        245000 non-null  int32         
 4   original_balance    245000 non-null  float64       
 5   current_balance_bk  245000 non-null  float64       
 6   EAD                 245000 non-null  float64       
 7   credit_score_orig   245000 non-null  float64       
 8   loan_to_value_orig  245000 non-null  float64       
 9   interest_rate       245000 non-null  float64       
 10  product_type        245000 non-null  object        
 11  ever_defaulted      245000 non-null  bool          
 12  default_date        171157 non-null  datetime64[ns]
 13  PD                  245000 no

In [13]:
type( df_panel )

pandas.core.frame.DataFrame

# Step 2: Reculculate the current_balance column

Write a vectorized or row-wise function that — given the original balance, original rate, original term, age in months at that report date, and optional prepayment/default flags — computes the remaining balance at that point in time.



In [14]:
df_panel.columns

Index(['loan_id', 'origination_date', 'report_date', 'num_payments',
       'original_balance', 'current_balance_bk', 'EAD', 'credit_score_orig',
       'loan_to_value_orig', 'interest_rate', 'product_type', 'ever_defaulted',
       'default_date', 'PD', 'LGD', 'projected_loss', 'loan_term_months',
       'unemployment', 'gdp_growth_qoq', 'hpi_change', 'bbb_spread'],
      dtype='object')

### Step 2-A: Use top 10 rows to test amortization logic

In [15]:
df_panel_qc = df_panel.head( 10 )

In [16]:
# ── The function ────────────────────────────────────────────────
def add_amortizing_balance_simple(df):
    df = df.copy()
    df['months_elapsed'] = ((df['report_date'] - df['origination_date']).dt.days // 30).clip(lower=0)

    def balance_after_months(row):
        orig_bal = row['original_balance']
        r_annual = row['interest_rate']
        term     = row['loan_term_months']
        m        = row['months_elapsed']

        if m <= 0: return orig_bal
        if m >= term: return 0.0

        r = r_annual / 12 / 100
        if r < 1e-6:
            return orig_bal * (1 - m / term)

        print( "value of r is ---:", r )

        factor = ((1 + r)**term - (1 + r)**m) / ((1 + r)**term - 1)

        print( "value of factor is ---:", factor )

        return orig_bal * factor

    df['current_balance'] = df.apply(balance_after_months, axis=1).round(2)
    # df = df.drop(columns=['months_elapsed'], errors='ignore')
    return df

# ── Run it correctly ────────────────────────────────────────────
df_panel_qc = add_amortizing_balance_simple(df_panel_qc)

# Now this should work
print(df_panel_qc[['loan_id', 'origination_date', 'report_date', 'months_elapsed', 'original_balance', 'current_balance', 'current_balance_bk', 'interest_rate' ]])



value of r is ---: 0.002645
value of factor is ---: 0.9623241007107899
value of r is ---: 0.002645
value of factor is ---: 0.9605589870512855
value of r is ---: 0.002645
value of factor is ---: 0.9587892046661516
value of r is ---: 0.002645
value of factor is ---: 0.9570147412066091
value of r is ---: 0.002645
value of factor is ---: 0.9552355842912161
value of r is ---: 0.002645
value of factor is ---: 0.9534517215057818
value of r is ---: 0.002645
value of factor is ---: 0.9516631404032802
value of r is ---: 0.002645
value of factor is ---: 0.9498698285037622
value of r is ---: 0.002645
value of factor is ---: 0.9480717732942702
value of r is ---: 0.002645
value of factor is ---: 0.946268962228749
   loan_id origination_date report_date  months_elapsed  original_balance  \
0        1       2016-03-31  2018-01-31              22         541879.93   
1        1       2016-03-31  2018-02-28              23         541879.93   
2        1       2016-03-31  2018-03-31              24     

### Step 2-B: Run amortization logic on full data

In [17]:
# ── The function ────────────────────────────────────────────────
def add_amortizing_balance_simple(df):
    df = df.copy()
    df['months_elapsed'] = ((df['report_date'] - df['origination_date']).dt.days // 30).clip(lower=0)

    def balance_after_months(row):
        orig_bal = row['original_balance']
        r_annual = row['interest_rate']
        term     = row['loan_term_months']
        m        = row['months_elapsed']

        if m <= 0: return orig_bal
        if m >= term: return 0.0

        r = r_annual / 12 / 100
        if r < 1e-6:
            return orig_bal * (1 - m / term)

        factor = ((1 + r)**term - (1 + r)**m) / ((1 + r)**term - 1)

        return orig_bal * factor

    df['current_balance'] = df.apply(balance_after_months, axis=1).round(2)
    # df = df.drop(columns=['months_elapsed'], errors='ignore')
    return df

# ── Run it correctly ────────────────────────────────────────────
df_panel = add_amortizing_balance_simple( df_panel )

# Now this should work
print(df_panel[['loan_id', 'origination_date', 'report_date', 'months_elapsed', 'original_balance', 'current_balance', 'current_balance_bk', 'interest_rate' ]].head( 5 ))



   loan_id origination_date report_date  months_elapsed  original_balance  \
0        1       2016-03-31  2018-01-31              22         541879.93   
1        1       2016-03-31  2018-02-28              23         541879.93   
2        1       2016-03-31  2018-03-31              24         541879.93   
3        1       2016-03-31  2018-04-30              25         541879.93   
4        1       2016-03-31  2018-05-31              26         541879.93   

   current_balance  current_balance_bk  interest_rate  
0        521464.12           430203.61          3.174  
1        520507.64           430203.61          3.174  
2        519548.63           430203.61          3.174  
3        518587.08           430203.61          3.174  
4        517622.99           430203.61          3.174  


In [18]:
### output to a local directory

df_panel.to_csv( myout + "/CCAR_simulated_data_20260318_01.csv" )

df_qc= pd.read_csv( myout + "/CCAR_simulated_data_20260318_01.csv" )

df_qc.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245000 entries, 0 to 244999
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          245000 non-null  int64  
 1   loan_id             245000 non-null  int64  
 2   origination_date    245000 non-null  object 
 3   report_date         245000 non-null  object 
 4   num_payments        245000 non-null  int64  
 5   original_balance    245000 non-null  float64
 6   current_balance_bk  245000 non-null  float64
 7   EAD                 245000 non-null  float64
 8   credit_score_orig   245000 non-null  float64
 9   loan_to_value_orig  245000 non-null  float64
 10  interest_rate       245000 non-null  float64
 11  product_type        245000 non-null  object 
 12  ever_defaulted      245000 non-null  bool   
 13  default_date        171157 non-null  object 
 14  PD                  245000 non-null  float64
 15  LGD                 245000 non-nul

In [19]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")

Started:  2026-03-18 13:19:46.987586-04:00
Finished: 2026-03-18 13:20:03.972012-04:00

Duration: 0:00:16.984426
Duration: 16.984 seconds
